# 03 — URL model training

`URLCharModel`: char embedding → multi-kernel Conv1d → BiLSTM → additive attention
pooling → classifier, with a parallel handcrafted-feature branch.

From `app/models/url_model.py:60`.

> **Trap.** `use_handcrafted=True` is silently downgraded to off when
> `n_handcrafted=0` (line 86). The branch only activates if you pass
> `n_handcrafted=30`. If you skip this, your 30 features are computed and then
> thrown away.

In [ ]:
import sys
from pathlib import Path

BACKEND = Path(r"E:\UIU\TRIMESTER - 11\Computer Security\CS_Project\Backend")
DATASET = BACKEND.parent / "Dataset"
if str(BACKEND) not in sys.path:
    sys.path.insert(0, str(BACKEND))

import json
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)

SEED = 42
np.random.seed(SEED)

print("backend :", BACKEND)
print("dataset :", DATASET)
print("numpy   :", np.__version__)
print("pandas  :", pd.__version__)

In [ ]:
def show(title, obj):
    print(f"\n=== {title} ===")
    print(obj)

def hr(title):
    print("\n" + "=" * 70)
    print(title)
    print("=" * 70)

NOTEBOOK_DIR = BACKEND / "notebooks"
ARTIFACT_DIR = NOTEBOOK_DIR / "artifacts"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

def save_artifact(name, obj):
    """Persist a JSON-serialisable object and echo where it landed."""
    path = ARTIFACT_DIR / name
    path.write_text(json.dumps(obj, indent=2, default=str), encoding="utf-8")
    print(f"saved -> {path}")
    return path

## 3.1 Config

In [ ]:
from training.train_url import load_config, resolve_paths, set_seed, class_weights

CFG_PATH = BACKEND / "configs" / "dev.yaml"
cfg = load_config(CFG_PATH)
paths = resolve_paths(cfg, CFG_PATH)

set_seed(SEED)
um = cfg["url_model"]

show("resolved paths", {k: str(v) for k, v in paths.items()})
show("url_model config", {k: um.get(k) for k in
    ("max_length","max_train_rows","batch_size","epochs","lr","handcrafted_branch","dropout")})

## 3.2 Data

In [ ]:
from app.preprocessing.url_dataset import load_splits, subsample, URLDataset, make_loader

splits = load_splits(paths["splits_dir"])
train_sp = subsample(splits["train"], um.get("max_train_rows"), seed=SEED)
val_sp, test_sp = splits["val"], splits["test"]

for nm, sp in (("train", train_sp), ("val", val_sp), ("test", test_sp)):
    print(f"{nm:>5}: n={len(sp):>7,}  phishing_rate={sp.labels.mean():.4f}")

In [ ]:
from app.preprocessing.url_preprocessing import CharTokenizer
from app.preprocessing.url_features import FeatureScaler, extract_batch, FEATURE_NAMES

MAX_LEN = um.get("max_length", 256)

tok = CharTokenizer(max_length=MAX_LEN, min_count=5).fit(train_sp.urls)

# Fit the scaler on the SUBSAMPLED train set only.
scaler = FeatureScaler().fit(extract_batch(train_sp.urls))

train_ds = URLDataset(train_sp, tok, scaler)
val_ds   = URLDataset(val_sp,   tok, scaler)
test_ds  = URLDataset(test_sp,  tok, scaler)

show("dataset tensors", {
    "char_ids": tuple(train_ds.char_ids.shape),
    "mask": tuple(train_ds.mask.shape),
    "handcrafted": tuple(train_ds.handcrafted.shape),
    "labels": tuple(train_ds.labels.shape),
})

## 3.3 Instantiate — with the handcrafted branch actually on

In [ ]:
import torch
from app.models.url_model import URLCharModel

N_HAND = len(FEATURE_NAMES)

model = URLCharModel(
    vocab_size=tok.vocab_size,
    max_length=MAX_LEN,
    n_handcrafted=N_HAND,
    use_handcrafted=True,
    embedding_out=128,
    dropout=um.get("dropout", 0.35),
)

show("handcrafted branch active", model.use_handcrafted)
assert model.use_handcrafted is True, "handcrafted branch silently disabled"

n_params = sum(p.numel() for p in model.parameters())
print(f"parameters: {n_params:,}")

## 3.4 Forward-pass sanity check

In [ ]:
batch = next(iter(make_loader(train_ds, 8, shuffle=False)))
char_ids, mask, hand, y = batch
print("shapes:", tuple(char_ids.shape), tuple(mask.shape), tuple(hand.shape), tuple(y.shape))

model.eval()
with torch.no_grad():
    out = model(char_ids, mask, hand)

show("output", {
    "logit": tuple(out.logit.shape),
    "probability": tuple(out.probability.shape),
    "embedding": tuple(out.embedding.shape),
    "attention_rows_sum_to_1": [round(float(s), 4) for s in out.attention.sum(1)[:4]],
})
assert torch.allclose(out.attention.sum(1), torch.ones(8), atol=1e-4)
print("attention weights normalise correctly")

## 3.5 Class weighting — use the project's `WeightedBCE`, not `pos_weight`

`class_weights()` returns a **2-element** `[1.0, n_neg/n_pos]` tensor of *per-class*
weights. Passing that as `BCEWithLogitsLoss(pos_weight=...)` is a shape error, and it
silently mis-weights even where shapes happen to line up. `WeightedBCE` applies the
weight per sample, which is what inverse-frequency weighting actually requires.

In [ ]:
from training.train_url import WeightedBCE, class_weights

w = class_weights(train_sp.labels)
show("class_weights", {"tensor": w.tolist(), "shape": tuple(w.shape)})
print("This is a per-class weight vector, not a scalar pos_weight.")

In [ ]:
from datetime import datetime
import torch
from torch import nn

DEVICE = torch.device("cpu")
model.to(DEVICE)

BATCH = um.get("batch_size", 32)
EPOCHS = 3  # dev config is 5; 3 keeps the notebook runtime tolerable

train_loader = make_loader(train_ds, BATCH, shuffle=True, seed=SEED)
val_loader   = make_loader(val_ds,   256, shuffle=False)

opt = torch.optim.AdamW(model.parameters(), lr=um.get("lr", 1e-3), weight_decay=1e-4)

# The same criterion the production trainer uses.
crit = WeightedBCE(pos_weight=float(class_weights(train_sp.labels)[1])).to(DEVICE)
loss_fn = crit

history = []
for epoch in range(1, EPOCHS + 1):
    model.train()
    total = n = 0
    for cb, mb, hb, yb in train_loader:
        cb, mb, hb, yb = cb.to(DEVICE), mb.to(DEVICE), hb.to(DEVICE), yb.to(DEVICE)
        opt.zero_grad()
        loss = crit(model(cb, mb, hb).logit, yb)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        total += float(loss) * len(yb)
        n += len(yb)

    model.eval()
    vp = vn = 0.0
    with torch.no_grad():
        for cb, mb, hb, yb in val_loader:
            cb, mb, hb, yb = cb.to(DEVICE), mb.to(DEVICE), hb.to(DEVICE), yb.to(DEVICE)
            vp += float(crit(model(cb, mb, hb).logit, yb)) * len(yb)
            vn += len(yb)

    train_loss, val_loss = total / n, vp / vn
    history.append({"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss})
    print(f"epoch {epoch}/{EPOCHS}  train_loss={train_loss:.4f}  val_loss={val_loss:.4f}  ({datetime.now():%H:%M:%S})")

## 3.6 Persist

In [ ]:
OUT = ARTIFACT_DIR / "url_model_notebook.pt"
torch.save({
    "state_dict": model.state_dict(),
    "model_config": model.config(),
    "tokenizer_state": tok.state_dict(),
    "scaler_state": scaler.state_dict(),
    "history": history,
    "seed": SEED,
}, OUT)
show("saved", {"path": str(OUT), "size_mb": round(OUT.stat().st_size / 1e6, 2)})
show("model_config", model.config())

> **Note.** These notebooks *call* the production modules in `app/` and
> `training/`. They do not reimplement the pipeline. The logic under test lives
> in exactly one place, so a notebook can never drift from what the CLI scripts
> and the API actually run.

## Next

`04_evaluation_and_leakage.ipynb` — metrics, calibration, leakage comparison.